# Mandarin (en-zh) fine-tune on Colab GPU

Before running: in VS Code, click **Select Kernel** (top-right) -> **Colab** -> choose a GPU runtime (e.g. T4).

**One-time manual step, do this first**: upload the 4 processed data files
(`data/processed/en-zh/{train,val}.{en,zh}`, ~22MB total, from the local repo)
to a Google Drive folder, e.g. `MyDrive/medical_vlm_data/en-zh/`. Update
`DRIVE_DATA_DIR` below to match wherever you put them.

Checkpoints are written to Drive (not the ephemeral Colab VM disk), so if
the session disconnects (free tier: ~90 min idle / ~12h max), just
reconnect and re-run this notebook top to bottom -- the last cell
auto-detects an existing checkpoint and resumes from it instead of
starting over.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- stop and pick a GPU runtime')
assert torch.cuda.is_available(), 'No GPU attached -- use Select Kernel -> Colab -> GPU runtime'

In [ ]:
# Code only (no data) -- data comes from Drive in the next cells
REPO_URL = "https://github.com/REZZOUGAICHA/Medical_VLM.git"

!git clone $REPO_URL repo
%cd repo

In [ ]:
# Deliberately NOT reinstalling torch -- Colab's preinstalled torch already has
# CUDA wired up correctly; reinstalling from requirements.txt risks silently
# swapping it for a CPU-only build.
!pip install -q transformers sentencepiece sacrebleu PyYAML accelerate sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE_DATA_DIR = "/content/drive/MyDrive/medical_vlm_data/en-zh"  # where you uploaded train/val.{en,zh}
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-zh-medical"

assert os.path.isdir(DRIVE_DATA_DIR), f"{DRIVE_DATA_DIR} not found -- upload the processed data there first"
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    assert os.path.isfile(os.path.join(DRIVE_DATA_DIR, f)), f"missing {f} in {DRIVE_DATA_DIR}"

os.makedirs("data/processed/en-zh", exist_ok=True)
for f in ["train.en", "train.zh", "val.en", "val.zh"]:
    shutil.copy(os.path.join(DRIVE_DATA_DIR, f), os.path.join("data/processed/en-zh", f))

# models/ -> Drive, so checkpoints (and the final model) survive a disconnect
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
if os.path.islink("models") or os.path.isdir("models"):
    if os.path.islink("models"):
        os.remove("models")
    else:
        shutil.rmtree("models")
os.symlink("/content/drive/MyDrive/medical_vlm_checkpoints", "models")
print("Data and checkpoint dir ready.")

In [ ]:
import glob

checkpoints = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINT_DIR, "checkpoint-*")), key=os.path.getmtime)
resume_flag = f'--resume-from-checkpoint "{checkpoints[-1]}"' if checkpoints else ""
if checkpoints:
    print(f"Resuming from {checkpoints[-1]}")
else:
    print("No existing checkpoint found -- starting fresh.")

!python src/train.py --config configs/en-zh.yaml {resume_flag}